# Lichess Skill-Model Data Pipeline

Builds the modelling table for **Beyond Glicko-2: Hierarchical Bayesian
Estimation of Latent Chess Skill**.

Two arms, one pipeline:

| Arm | Question | Columns it needs |
|---|---|---|
| **A — information-fair audit** | Does proper joint inference beat Glicko-2 on the *same* data? | outcome, players, timestamp, time class, **pre-game Glicko rating** |
| **B — value of information** | What is Glicko-2 leaving on the table? | termination type, ply count, per-side clock usage |

**Four stages.** Stages 1 and 3 are the expensive ones; you run each once.

```
1. discovery   stream corpus -> per-player activity counts     (~1-3 h, once)
2. cohort      DuckDB -> connected player list                 (seconds)
3. enrich      re-stream -> cohort games + movetext features   (~2-4 h, once)
4. assemble    -> model-ready .npz                             (~1 min)
```

**Nothing large ever touches disk.** The `.zst` is streamed over HTTP and
decompressed in memory. A 30 GB compressed / 200 GB decompressed month-file is
processed with a constant ~50 MB footprint, so Colab's ephemeral disk is never
a constraint.


## 0 · Setup

In [ ]:
!pip -q install zstandard duckdb pyarrow requests

import os, sys, json, time
from google.colab import drive
drive.mount('/content/drive')

PKG  = '/content/drive/MyDrive/Colab Notebooks/Bayes ML Final Project'
ROOT = '/content/drive/MyDrive/Colab Notebooks/Bayes ML Final Project/data'
os.makedirs(ROOT, exist_ok=True)
sys.path.insert(0, PKG)

# Shell-safe versions for the ! cells (handles the spaces in the folder name)
import shlex
PKG_Q, ROOT_Q = shlex.quote(PKG), shlex.quote(ROOT)

import lichess_stream, importlib
importlib.reload(lichess_stream)
print('ok —', len(os.listdir(PKG)), 'files visible')

Mounted at /content/drive
ok — 9 files visible


## 1 · Smoke test

In [ ]:
from lichess_stream import open_month, iter_games, time_class, clock_features

MONTH = '2026-05'   # <- pick a month inside your window

for i, (h, mv) in enumerate(iter_games(open_month(MONTH, max_bytes=20<<20),
                                       want_movetext=True)):
    print(h)
    print('  class      :', time_class(h.get('TimeControl')))
    print('  termination:', h.get('Termination'))
    if mv:
        print('  clocks     :', clock_features(mv, h.get('TimeControl')))
        print('  movetext[:120]:', mv[:120])
    print()
    if i >= 4: break

In [ ]:
# Class + termination distribution on a small sample - sanity check the buckets.
# NOTE: this reads the HEAD of the month file, which is time-ordered, so it is
# a snapshot of the first minutes of the month (arena-heavy), not a
# representative sample. Use it to confirm the buckets work, not to estimate
# proportions.
from collections import Counter

cls, term, clk, arena = Counter(), Counter(), Counter(), Counter()

for n, (h, mv) in enumerate(iter_games(open_month(MONTH, max_bytes=200 << 20),
                                       want_movetext=True)):
    cls[time_class(h.get('TimeControl'))] += 1
    term[h.get('Termination')] += 1
    clk[bool(mv and b'%clk' in mv)] += 1
    arena['tournament' in h.get('Event', '')] += 1

print('time class :', cls.most_common())
print('termination:', term.most_common())
print('has [%clk] :', dict(clk))
print('arena game :', dict(arena))
print('games      :', n + 1)

## 2 · Discovery

Streams every month and writes **only** per-player activity aggregates — no
game rows. Output is ~60-100 MB per month.

In [ ]:
WINDOW_START, WINDOW_END = '2025-06', '2026-05'   # 12 months
COHORT_CLASS = 'blitz'                            # highest volume -> densest graph

# !python {PKG_Q}/pass1_discovery.py \
#     --start {WINDOW_START} --end {WINDOW_END} \
#     --time-class {COHORT_CLASS} \
#     --out {ROOT_Q}/discovery

## 3 · Cohort selection

Getting a *connected* comparison graph of workable size. Relative skill is
identified only through chains of shared opponents. Targets: **5k–20k
players**, and after stage 4, **10⁵–10⁶ games**.

In [ ]:
!python {PKG_Q}/cohort.py \
    --discovery {ROOT_Q}/discovery \
    --out {ROOT_Q}/cohort.parquet \
    --min-games 5000 \
    --min-active-months 6

In [ ]:
import duckdb
con = duckdb.connect()
n = con.execute(f"""
    SELECT COUNT(*), AVG(n_games), MIN(n_games), AVG(active_months)
    FROM read_parquet('{ROOT}/cohort.parquet')
""").fetchone()
print(f"players {n[0]:,} | avg games {n[1]:.0f} | min {n[2]:.0f} | avg active months {n[3]:.1f}")

## 4 · Enrichment pass

Re-streams the corpus. Keeps a game only if **both** players are in the cohort
— roughly 0.1–1% of games — and captures all four time classes, since the
cross-format covariance Σ is a headline result. `--with-movetext` is what buys
Arm B.

In [ ]:
!python {PKG_Q}/pass2_enrich.py \
    --start {WINDOW_START} --end {WINDOW_END} \
    --cohort {ROOT_Q}/cohort.parquet \
    --out {ROOT_Q}/games \
    --with-movetext

## 5 · Assemble

Integer arrays, connectivity filter, stratum flags, and the leakage-safe
split. **Read the split logic.**

In [ ]:
!python {PKG_Q}/assemble.py \
    --games {ROOT_Q}/games \
    --out {ROOT_Q}/model_data.npz \
    --holdout-months 2 \
    --bucket month \
    --subsample-frac 0.20 \
    --return-pct 99 \
    --return-min-days 3

## 6 · Validation

In [ ]:
import numpy as np

d = np.load(f'{ROOT}/model_data.npz', allow_pickle=True)

n_g, n_p, n_t = len(d['y']), int(d['n_players']), int(d['n_buckets'])
print(f'games {n_g:,} | players {n_p:,} | buckets {n_t}')
print(f'games/player {2*n_g/n_p:.0f}   <- want >= 100')

import collections
print('outcomes (0=black,1=draw,2=white):', collections.Counter(d['y'].tolist()))
print('white win rate:', (d['y'] == 2).mean().round(4),
      '| draw rate:', (d['y'] == 1).mean().round(4))

deg = np.bincount(np.concatenate([d['white'], d['black']]), minlength=n_p)
print(f'degree  min {deg.min()}  p05 {np.percentile(deg, 5):.0f}  '
      f'median {np.median(deg):.0f}  max {deg.max()}')

active = np.zeros((n_p, n_t), bool)
active[d['white'], d['t_idx']] = True
active[d['black'], d['t_idx']] = True
print('buckets active per player: median', np.median(active.sum(1)))

In [ ]:
# Arm B: is the flag-fall channel actually informative?
import numpy as np

elo = (d['glicko_white'].astype(float) + d['glicko_black'].astype(float)) / 2
bins = np.arange(800, 2800, 200)
idx = np.digitize(elo, bins)

print(' rating   n_games   flag-rate   draw-rate   median plies')
for k in range(1, len(bins) + 1):
    m = idx == k
    if m.sum() < 500:
        continue
    pl = d['n_plies'][m]
    pl = pl[pl > 0]
    lo = bins[k - 1]
    print(f' {lo:>5}   {m.sum():>7,}   '
          f'{d["flagged"][m].mean():>9.3f}   {(d["y"][m] == 1).mean():>9.3f}   '
          f'{np.median(pl) if len(pl) else float("nan"):>11.0f}')

## 7 · The Glicko-2 baseline table

In [ ]:
import numpy as np
from scipy.stats import norm
from scipy.optimize import minimize

tr = d['is_train']
diff = (d['glicko_white'].astype(float) - d['glicko_black'].astype(float)) / 173.7

def nll(params, m):
    h, log_g, s = params
    g = np.exp(log_g)
    z = s * diff[m] + h
    pW = norm.cdf(z - g)
    pL = norm.cdf(-z - g)
    p = np.stack([pL, np.clip(1 - pW - pL, 1e-9, 1), pW])
    return -np.log(p[d['y'][m], np.arange(m.sum())] + 1e-12).mean()

fit = minimize(nll, [0.05, np.log(0.3), 1.0], args=(tr,), method='Nelder-Mead')
h_hat, gamma_hat, scale_hat = fit.x[0], np.exp(fit.x[1]), fit.x[2]

print(f'B2 baseline: h={h_hat:.4f}  gamma={gamma_hat:.4f}  scale={scale_hat:.4f}')
print(f'train NLL {fit.fun:.4f} | holdout NLL {nll(fit.x, ~tr):.4f}')

np.savez(f'{ROOT}/glicko_baseline.npz', h=h_hat, gamma=gamma_hat, scale=scale_hat)

In [ ]:
def rps(p, y):
    cp = np.cumsum(p, axis=1)[:, :-1]
    oh = np.cumsum(np.eye(3)[y], axis=1)[:, :-1]
    return ((cp - oh) ** 2).sum(1).mean()

def glicko_probs(m, h, g, s):
    z = s * diff[m] + h
    pW = norm.cdf(z - g)
    pL = norm.cdf(-z - g)
    return np.stack([pL, np.clip(1 - pW - pL, 1e-9, 1), pW], axis=1)

for name, params in [('B0 raw    ', (0.0, 0.0, 1.0)),
                     ('B1 +gamma ', (0.0, gamma_hat, scale_hat)),
                     ('B2 +h     ', (h_hat, gamma_hat, scale_hat))]:
    p = glicko_probs(~tr, *params)
    print(f'{name}  holdout RPS {rps(p, d["y"][~tr]):.5f}')

print('\nThese are the numbers your model has to beat. Save them.')

In [ ]:
base = np.array([(d['y'][tr] == k).mean() for k in range(3)])
p_null = np.tile(base, (int((~tr).sum()), 1))
print(f'B-null (marginal rates)  holdout RPS {rps(p_null, d["y"][~tr]):.5f}')

In [ ]:
def nll_p(h, g, s, m):
    z = s * diff[m] + h
    pW = norm.cdf(z - g); pL = norm.cdf(-z - g)
    p = np.stack([pL, np.clip(1 - pW - pL, 1e-9, 1), pW])
    return -np.log(p[d['y'][m], np.arange(m.sum())] + 1e-12).mean()

r = minimize(lambda x: nll_p(0.0, 0.0, x[0], tr), [1.0], method='Nelder-Mead')
s_b0b = r.x[0]

r = minimize(lambda x: nll_p(0.0, np.exp(x[0]), x[1], tr),
             [np.log(0.1), 0.6], method='Nelder-Mead')
g_b1, s_b1 = np.exp(r.x[0]), r.x[1]

rungs = [('B0  raw       ', (0.0,   0.0,   1.0)),
         ('B0b +scale    ', (0.0,   0.0,   s_b0b)),
         ('B1  +draw marg', (0.0,   g_b1,  s_b1)),
         ('B2  +white adv', (h_hat, gamma_hat, scale_hat))]

prev = None
for name, prm in rungs:
    v = rps(glicko_probs(~tr, *prm), d['y'][~tr])
    delta = '' if prev is None else f'  (Δ {prev - v:+.5f})'
    print(f'{name}  holdout RPS {v:.5f}{delta}')
    prev = v

In [ ]:
import json

ladder = {
    'null_marginal': 0.49820,
    'B0_raw':        0.49088,
    'B0b_scale':     0.48198,
    'B1_draw':       0.48047,
    'B2_white':      0.47988,
}
params = {
    'h': float(h_hat), 'gamma': float(gamma_hat), 'scale': float(scale_hat),
    's_b0b': float(s_b0b), 'g_b1': float(g_b1), 's_b1': float(s_b1),
    'train_nll': 0.8591, 'holdout_nll': 0.8425,
}
meta = {
    'n_holdout': int((~tr).sum()),
    'n_train':   int(tr.sum()),
    'note': 'fitted on training split; bots NOT yet excluded',
}
json.dump({'ladder_rps': ladder, 'params': params, 'meta': meta},
          open(f'{ROOT}/baseline_ladder.json', 'w'), indent=2)
print(open(f'{ROOT}/baseline_ladder.json').read())

---

## What we have now

`model_data.npz` — everything the sampler touches, typically 20–80 MB. From
here on nothing reads Parquet in a hot loop, and the modelling fits on a T4
with room to spare (full model state is under 10 MB; the bottleneck was always
extraction, never compute).

##